In [0]:
%pip install --extra-index-url=https://pypi.nvidia.com cuopt-server-cu12 cuopt-sh-client
%restart_python

In [0]:
NUM_SHIPMENTS = 40_000
NUM_ROUTES = 320  # total trucks available
MAX_EV = 4000 # max capacity
MAX_VAN = 8000
DEPOT_LAT, DEPOT_LON = 39.7685, -86.1580 # start and end point for each route


catalog = "default"
schema = "routing"
shipments_table = f"{catalog}.{schema}.raw_shipments"
clustered_table = f"{catalog}.{schema}.shipment_clusters_gpu"
distances_table = f"{catalog}.{schema}.distances_by_route_gpu"
routing_table = f"{catalog}.{schema}.optimized_routes_gpu"

In [0]:
%sh nvidia-smi

In [0]:
import cudf
from cuopt import routing
import pandas as pd
import numpy as np
from pyspark.sql import functions as F

In [0]:
cost = cudf.DataFrame([[0,3,1,2],[3,0,1,2],[2,3,0,2],[2,3,1,0]], dtype='float32')
n_locations = cost.shape[0]
n_vehicles = 2
n_orders = 3  # one order per task node

dm = routing.DataModel(n_locations, n_vehicles, n_orders)
dm.add_cost_matrix(cost)
dm.add_transit_time_matrix(cost.copy(deep=True))  # separate if times differ

ss = routing.SolverSettings()
ss.set_verbose_mode(True)
# ss.set_time_limit(5)
sol = routing.Solve(dm, ss)

print(sol.get_route())      # pandas-like table
sol.display_routes()        # pretty print

In [0]:
catalog = "default"
schema = "routing"
shipments_table = f"{catalog}.{schema}.raw_shipments"
distances_table = f"{catalog}.{schema}.distances_by_route_gpu"
distances_df = spark.read.table(distances_table)
display(distances_df.orderBy("origin_id", "destination_index")) # TODO change order

In [0]:
# === Build a cuOpt-ready cost matrix from distances_df (cost = duration_seconds) ===
# 1) Get the node list (deterministic order)
nodes = (
    distances_df
      .select(F.col("global_idx_source").alias("id"))
      .unionByName(distances_df.select(F.col("global_idx_dest").alias("id")))
      .distinct()
      .orderBy("id")
      .toPandas()["id"]
      .tolist()
)

# If you have a known depot global index, put it first:
nodes.sort(key=lambda x: (x != 0, x))

n = len(nodes)
idx_pos = {g:i for i,g in enumerate(nodes)}

# 2) Initialize matrix with a big finite penalty; 0 on diagonal
M = np.full((n, n), 1e9, dtype=np.float32)
np.fill_diagonal(M, 0.0)

# 3) Fill matrix from the distances table
pdf = (
    distances_df
      .select("global_idx_source", "global_idx_dest", F.col("duration_seconds").alias("cost"))
      .toPandas()
)

for s, d, c in pdf.itertuples(index=False):
    i = idx_pos[s]; j = idx_pos[d]
    M[i, j] = np.float32(c)

cost_gdf = cudf.DataFrame(M)

# 4) cuOpt model: cost = distance, reuse as transit time (simple case)
n_locations = n
n_orders = max(n_locations - 1, 0)  # tasks = all non-depot nodes if you use a depot-first convention

dm = routing.DataModel(n_locations, NUM_ROUTES, n_orders)
dm.add_cost_matrix(cost_gdf)
dm.add_transit_time_matrix(cost_gdf)

ss = routing.SolverSettings()
ss.set_time_limit(60*20)  # time limit in seconds
sol = routing.Solve(dm, ss)

# 5) (Optional) map cuOpt node_index -> your global_idx
route_pdf = sol.get_route().to_pandas()
route_pdf["global_idx"] = route_pdf["node_index"].map(lambda i: nodes[int(i)])

In [0]:
optimized_routes_df = spark.createDataFrame(route_pdf)
optimized_routes_df.write.saveAsTable(routing_table)

In [0]:
spark.read.table(routing_table).display()